# 03 — Genie on Raw Tables vs. Genie on a Metric View

**Time: about 12 minutes.**
Covers: The same question to two agents · Reading generated SQL · Why raw tables and metric views should not share an agent · Agents as code

### What you will be able to do afterwards

- Show, with numbers, what a semantic layer changes for a natural-language interface.
- Read Genie's generated SQL and name the definition it silently chose.
- Explain why mixing raw fact tables and metric views in one agent produces bad joins.

You will build the agent most teams build first: every table that looks relevant, no instructions.
Then ask it and your Exercise 2 agent the same ambiguous questions.

In [ ]:
from helpers import utils, genie_agents, observations, ground_truth

cfg = utils.use_lab_schema()
spark = utils.spark
print(f"Working in {cfg['catalog']}.{cfg['schema']}")

## Step 1 — Build the raw-tables agent

In [ ]:
print(f"Title: {genie_agents.suggested_title('raw_tables_agent')}\n")
print("Data sources (all five, and NOT the metric view):")
for v in ("lab_orders", "lab_refunds", "lab_customers", "lab_products", "lab_daily_sales_summary"):
    print(f"  {utils.fq_plain(v)}")

1. **Genie Agents > New**. Select the five views above. Then **Create**.
2. If Genie Code offers suggested context, **dismiss it** for this agent: the point is to see what
   Genie does with raw tables and nothing else.
3. **Configure > Scroll Down > About this agent**: Edit Agent Name as printed in the previous cell, same warehouse as before. No instructions, no example
   SQL, no benchmarks. Save it.
4. From **Configure > About this agent > Agent ID**: Copy the ID and paste in the next cel to register it:

In [ ]:
RAW_AGENT_ID = ""  # paste the ID, or leave empty to auto-discover by title

genie_agents.register("raw_tables_agent", RAW_AGENT_ID)

## Step 2 — Ask both agents the same two questions

The goal is to compare how the two agents answer identical questions. One agent uses the governed
metric view, this is the agent we built in notebook 2. The other uses raw tables with no guidance.

The next cell prints the two questions. Ask each question to **both** agents:

| Agent | Built in | Data it uses |
|---|---|---|
| `metric_view_agent` | Notebook 02 | The metric view |
| `raw_tables_agent` | Step 1 above | Five raw tables |

That gives you **four answers** in total (2 agents × 2 questions).

In [ ]:
for key in ("revenue_90d", "active_customers"):
    print(observations.QUESTIONS[key])

Record all four answers. `answer_value` is the number Genie showed; `generated_sql` is its SQL,
pasted verbatim (the checks re-run the metric-view agent's SQL and compare).

**For each answer:**

1. Open a **new chat** with the agent. Don't reuse a chat, because earlier messages change how
   Genie interprets later questions.
2. Paste the question **exactly as printed**. Don't reword it. If the wording changes, you can't tell
   whether a different answer came from the agent or from the phrasing.
3. Note the **number** Genie returns.
4. Open the generated SQL (**Show code** / **View SQL**) and copy it **as is**.

You'll paste the number and the SQL into the recording cell below. The number tells you *what* each
agent answered. The SQL tells you *why*: which filters, tables, and definitions it chose without being told.

In [ ]:
observations.record("03", "compare", "metric_view_agent", "revenue_90d",
    answer_value=None,      # TO DO
    generated_sql="""
""",
    notes="")

observations.record("03", "compare", "metric_view_agent", "active_customers",
    answer_value=None,      # TO DO
    generated_sql="""
""",
    notes="")

observations.record("03", "compare", "raw_tables_agent", "revenue_90d",
    answer_value=None,      # TO DO
    generated_sql="""
""",
    notes="")

observations.record("03", "compare", "raw_tables_agent", "active_customers",
    answer_value=None,      # TO DO
    generated_sql="""
""",
    notes="")

**Optional: let the Conversation API do the asking.** Set `AUTO_RECORD = True` to ask both agents
programmatically and record the results (it overwrites the cell above). Still open the UI and read
the SQL: that is the point of the exercise.

In [ ]:
AUTO_RECORD = False

if AUTO_RECORD:
    for role in ("metric_view_agent", "raw_tables_agent"):
        for key in ("revenue_90d", "active_customers"):
            r = genie_agents.ask(genie_agents.registered_id(role), observations.QUESTIONS[key])
            print(f"{role:20s} {key:18s} {r['status']:10s} {r['first_value']}")
            observations.record("03", "compare", role, key, answer_value=r["first_value"],
                                generated_sql=r["sql"] or "", notes="recorded via Conversation API")

## Step 3 — What did each agent actually compute?

In [ ]:
observations.diagnose_revenue()
display(observations.show("03"))

Read the raw agent's SQL with the Exercise 1 traps in mind:

- Does it select `revenue` from `lab_daily_sales_summary`? That column includes cancelled and test
  orders, and the job feeding it stopped three days ago. Its table comment just says *Daily revenue summary*.
- Does it join `lab_refunds` to `lab_orders` directly? That is the fan-out from Exercise 1.
- Does it filter `is_test_order`, or `status`? Nothing told it to.
- For active customers, which window did it pick, and why that one?

The raw agent is not "dumb". It made reasonable guesses; there was just nothing to tell it which guess
the company considers correct. The metric-view agent never had to guess.

## Step 4 — Why not give one agent both?

It is tempting to attach the metric view *and* the raw tables "for flexibility". It backfires:

- **Two definitions, one agent.** `net_revenue` in the metric view and `order_total` plus `lab_refunds`
  in the raw tables are both plausible routes to "revenue". Genie picks per question, so answers become
  inconsistent again, which is the problem the semantic layer exists to remove.
- **Metric views cannot be joined directly.** To combine one with a fact table Genie must aggregate in a
  CTE and join the result. When it instead joins raw facts at different grains (orders to refund events,
  orders to a daily summary) totals are silently duplicated.
- **Measures lose their filters.** Re-deriving revenue from `lab_orders` skips the `NOT is_test_order`
  filter and the cancelled exclusion that live inside the metric view.

A small lookup at a dimension's grain (one row per region, like `lab_region_targets`) is the safe
exception: it cannot fan out a metric that is already aggregated by region.

*Optional experiment.* Clone your metric-view agent (kebab menu > **Clone**), add `lab_orders` and
`lab_refunds` to the clone, ask the revenue question a few times in new chats, and register and record
it as `mixed_clone_agent`. Compare the SQL across runs.

---

## Challenge 3

Save a recommendation for your data platform team: which agent should business users get, why
(grounded in the numbers and SQL you saw), and one concrete change that would make the raw agent less wrong.

**TO DO**

In [ ]:
observations.save_recommendation(
    winner="",          # TO DO: the agent role whose revenue matched the governed number
    rationale="",       # TO DO: two or three sentences grounded in what you observed
    raw_agent_fix="",   # TO DO: one concrete change for the raw agent
)

## Checks

In [ ]:
from helpers import test_runner
test_runner.run("03-genie-vs-raw-tables")

---

## Appendix — The same agent as code (optional, "this exists too")

Everything you configured in the UI can live in Git. Genie Agents are a resource type in
**Declarative Automation Bundles** (the new name for Databricks Asset Bundles): `resources.genie_spaces`,
which requires Databricks CLI 1.3.0+ and the **direct** deployment engine. The agent's content is the
same `serialized_space` JSON the checks read, kept in a `*.geniespace.json` file. You can also
generate it from an existing agent with `databricks bundle generate genie-space`.

The cell below prints a bundle definition derived from your live agent. Nothing is deployed.

In [ ]:
genie_agents.export_bundle(genie_agents.registered_id("metric_view_agent"))

If you hand-write `serialized_space`, mind the validation rules: `version: 2`, every `id` is a
32-character lowercase hex string, and every list is pre-sorted (tables by identifier, instructions by id).
See the Genie Agents API documentation.